# 00 — DYAMOND data inventory

Survey the C1440-LLC2160 staging area and record, for the downstream notebooks:

1. the **ocean surface flux variables** available as raw MITgcm MDS binary under `mit/`
   (`oceQnet`, `oceQsw`, `oceFWflx`, `oceTAUX/Y`, …), their output cadence, and the
   time base (iteration 0 ↔ 2020-01-19 21:00 UTC, 45 s timestep — verify against
   `mit/readme.txt`);
2. the **GEOS atmosphere collections** (NetCDF under `holding/`) that carry the surface
   heat flux components — expected in `tavg_15mn_2d_flx_Mx` (turbulent) and
   `geosgcm_surf` (radiative) — and the exact variable names;
3. the grid descriptors (`mit/grid`, `holding/geos_c1440_lats_lons_2D.nc`).

Reference: Menemenlis et al. (2026), *Sci. Data* — GEOS c1440 (~7 km) coupled to MITgcm
LLC2160 (~2–4 km), 2020-01-20 to 2021-03-26.

In [ ]:
# Environment check: run on SciServer (Kraken domain, with the Poseidon DYAMOND
# ceph volume attached), or set DYAMOND_ROOT to a local subset.
from dyamond_fluxes import dyamond_root

root = dyamond_root()  # raises with setup instructions if the data is absent
print(f"DYAMOND root: {root}")

In [ ]:
# The staging area's readme documents conventions — read it first.
readme = root / "mit" / "readme.txt"
if readme.exists():
    print(readme.read_text()[:4000])

In [ ]:
from dyamond_fluxes import list_mit_variables

variables = list_mit_variables()
print(f"{len(variables)} MITgcm variables with .data files:")
print("  ", ", ".join(variables))

In [ ]:
# Output cadence and time base for one flux variable.
import numpy as np

from dyamond_fluxes import open_mds_variable

qnet = open_mds_variable("oceQnet")
print(qnet)
step = np.diff(qnet.iteration.values[:5])
print("\niteration step:", step, "-> output every", step[0] * 45 / 3600, "h")
print("time range:", qnet.time.values[0], "to", qnet.time.values[-1])

In [ ]:
from dyamond_fluxes import open_grid

grid = open_grid()
print(grid)
print("\nlon range:", float(grid.XC.min()), "to", float(grid.XC.max()))
print("lat range:", float(grid.YC.min()), "to", float(grid.YC.max()))

## GEOS atmosphere collections

List the collections and the variables inside the two flux-relevant candidates. The
decomposition in notebook 02 needs latent (`EFLUX`-like), sensible (`HFLUX`-like), and
net surface shortwave/longwave radiation (`SWGNT`/`LWGNT`-like) — note the exact names
and their `long_name` sign conventions below.

In [ ]:
from dyamond_fluxes import list_geos_collections

collections = list_geos_collections()
print(f"{len(collections)} GEOS collections:")
for c in collections:
    print("  ", c)

In [ ]:
from dyamond_fluxes import peek_variables

for coll in ["tavg_15mn_2d_flx_Mx", "geosgcm_surf", "inst_15mn_2d_asm_Mx", "geosgcm_turb"]:
    if coll in collections:
        print(f"=== {coll} ===")
        for name, long_name in peek_variables(coll).items():
            print(f"  {name:16s} {long_name}")
        print()

In [ ]:
from dyamond_fluxes import load_geos_coords

coords = load_geos_coords()
coords

## Findings to carry forward

Fill in after running:

- Ocean flux output cadence: `…` (iterations per dump × 45 s)
- Time base confirmed against readme: **yes/no**
- Latent heat variable + collection: `…`
- Sensible heat variable + collection: `…`
- Net surface SW / LW variables + collection: `…`
- GEOS coordinate names in `geos_c1440_lats_lons_2D.nc`: `…`

If a radiation variable is missing from every collection, the full decomposition in
notebook 02 degrades gracefully to whatever components exist; the non-solar residual in
notebook 01 needs the ocean side only.